In [19]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

In [20]:
SCRIPT_DIR = Path.cwd()

DATA_PATH = (SCRIPT_DIR / "../DATA/averages_year_month.csv").resolve()
OUTPUT_DIR = (SCRIPT_DIR / "../OUTPUT").resolve()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Reading data from:", DATA_PATH)
print("Saving outputs to:", OUTPUT_DIR)

Reading data from: /Users/gsundaram/Documents/DSP/DS-4002-Project-2/DATA/averages_year_month.csv
Saving outputs to: /Users/gsundaram/Documents/DSP/DS-4002-Project-2/OUTPUT


In [21]:
df = pd.read_csv(DATA_PATH)
df = df.sort_values(["year", "month"]).reset_index(drop=True)

df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))
df = df.set_index("date")

feature_cols = {
    "Acousticness": "avg_acousticness",
    "Speechiness": "avg_speechiness",
    "Valence": "avg_valence",
    "Tempo": "avg_tempo",
    "Duration": "avg_duration"
}

In [22]:
def run_adf_test(series):
    result = adfuller(series.dropna(), autolag="AIC")
    return {
        "adf_statistic": result[0],
        "adf_pvalue": result[1],
        "used_lag": result[2],
        "nobs": result[3]
    }

def fit_best_sarima(
    series,
    seasonal_period=12,
    p_range=range(0, 3),
    d_range=range(0, 2),
    q_range=range(0, 3),
    P_range=range(0, 2),
    D_range=range(0, 2),
    Q_range=range(0, 2)
):
    best_model = None
    best_order = None
    best_seasonal_order = None
    best_aic = np.inf
    best_bic = np.inf

    for p in p_range:
        for d in d_range:
            for q in q_range:
                for P in P_range:
                    for D in D_range:
                        for Q in Q_range:
                            if (p, d, q) == (0, 0, 0) and (P, D, Q) == (0, 0, 0):
                                continue

                            try:
                                model = SARIMAX(
                                    series,
                                    order=(p, d, q),
                                    seasonal_order=(P, D, Q, seasonal_period),
                                    enforce_stationarity=False,
                                    enforce_invertibility=False
                                )
                                results = model.fit(disp=False)

                                if results.aic < best_aic:
                                    best_model = results
                                    best_order = (p, d, q)
                                    best_seasonal_order = (P, D, Q, seasonal_period)
                                    best_aic = results.aic
                                    best_bic = results.bic

                            except Exception:
                                continue

    return best_model, best_order, best_seasonal_order, best_aic, best_bic

def get_seasonal_terms(model_result):
    params = model_result.params
    pvalues = model_result.pvalues

    seasonal_rows = []
    for name in params.index:
        lname = name.lower()

        if ("s." in lname or "seasonal" in lname) and "sigma2" not in lname:
            seasonal_rows.append({
                "term": name,
                "coefficient": params[name],
                "p_value": pvalues[name],
                "significant_05": pvalues[name] < 0.05
            })

    return pd.DataFrame(seasonal_rows)

def has_seasonal_structure(seasonal_order):
    P, D, Q, s = seasonal_order
    return (P > 0) or (D > 0) or (Q > 0)

def make_conclusion(seasonal_order, seasonal_df):
    seasonal_structure = has_seasonal_structure(seasonal_order)

    if seasonal_df.empty:
        num_sig = 0
    else:
        num_sig = int(seasonal_df["significant_05"].sum())

    if seasonal_structure and num_sig > 0:
        return "The best-fitting model includes seasonal parameters with s = 12, and at least one seasonal parameter is statistically significant. This supports presence of monthly seasonality!"
    elif seasonal_structure and num_sig == 0:
        return "The best-fitting model includes seasonal parameters with s = 12, but no seasonal coefficients are statistically significant. This suggests possible monthly seasonality, but evidence is weak."
    else:
        return "The best-fitting model doesn't show meaningful seasonal structure with s = 12. This doesn't support strong evidence of monthly seasonality."

In [23]:
summary_rows = []

for feature_name, col in feature_cols.items():
    print("\n" + "=" * 80)
    print(f"FEATURE: {feature_name}")
    print("=" * 80)

    series = df[col].asfreq("MS")

    adf_info = run_adf_test(series)
    print(f"ADF statistic: {adf_info['adf_statistic']:.4f}")
    print(f"ADF p-value : {adf_info['adf_pvalue']:.4f}")

    best_model, best_order, best_seasonal_order, best_aic, best_bic = fit_best_sarima(series)

    if best_model is None:
        print("No SARIMA model fit successfully.")
        summary_rows.append({
            "feature": feature_name,
            "best_order": None,
            "best_seasonal_order": None,
            "AIC": None,
            "BIC": None,
            "ADF_pvalue": adf_info["adf_pvalue"],
            "has_seasonal_structure": None,
            "num_significant_seasonal_terms": None,
            "conclusion": "No model fit successfully."
        })
        continue

    print("Best non-seasonal order:", best_order)
    print("Best seasonal order    :", best_seasonal_order)
    print(f"AIC                    : {best_aic:.3f}")
    print(f"BIC                    : {best_bic:.3f}")

    coef_table = pd.DataFrame({
        "coefficient": best_model.params,
        "p_value": best_model.pvalues
    })
    coef_table["significant_05"] = coef_table["p_value"] < 0.05

    seasonal_df = get_seasonal_terms(best_model)

    print("\nCoefficient table:")
    print(coef_table)

    print("\nSeasonal terms only:")
    if seasonal_df.empty:
        print("No explicit seasonal AR/MA terms found.")
    else:
        print(seasonal_df)

    conclusion = make_conclusion(best_seasonal_order, seasonal_df)

    print("\nConclusion:")
    print(conclusion)

    coef_outfile = OUTPUT_DIR / f"{feature_name.lower()}_sarima_coefficients.csv"
    coef_table.to_csv(coef_outfile)

    seasonal_outfile = OUTPUT_DIR / f"{feature_name.lower()}_seasonal_terms.csv"
    seasonal_df.to_csv(seasonal_outfile, index=False)

    summary_rows.append({
        "feature": feature_name,
        "best_order": str(best_order),
        "best_seasonal_order": str(best_seasonal_order),
        "AIC": best_aic,
        "BIC": best_bic,
        "ADF_pvalue": adf_info["adf_pvalue"],
        "has_seasonal_structure": has_seasonal_structure(best_seasonal_order),
        "num_significant_seasonal_terms": 0 if seasonal_df.empty else int(seasonal_df["significant_05"].sum()),
        "conclusion": conclusion
    })



FEATURE: Acousticness
ADF statistic: -3.9204
ADF p-value : 0.0019
Best non-seasonal order: (2, 1, 1)
Best seasonal order    : (1, 0, 1, 12)
AIC                    : -1187.647
BIC                    : -1162.718

Coefficient table:
          coefficient       p_value  significant_05
ar.L1        0.052755  2.823127e-01           False
ar.L2       -0.111773  1.373372e-02            True
ma.L1       -0.962969  0.000000e+00            True
ar.S.L12     0.949953  0.000000e+00            True
ma.S.L12    -1.015319  2.375381e-13            True
sigma2       0.004064  2.449695e-11            True

Seasonal terms only:
       term  coefficient       p_value  significant_05
0  ar.S.L12     0.949953  0.000000e+00            True
1  ma.S.L12    -1.015319  2.375381e-13            True

Conclusion:
The best-fitting model includes seasonal parameters with s = 12, and at least one seasonal parameter is statistically significant. This supports presence of monthly seasonality!

FEATURE: Speechiness
ADF s

In [24]:
summary_df = pd.DataFrame(summary_rows)

summary_outfile = OUTPUT_DIR / "sarima_seasonality_summary.csv"
summary_df.to_csv(summary_outfile, index=False)

print("\n" + "#" * 80)
print("FINAL SUMMARY")
print("#" * 80)
print(summary_df)

print("\nSaved summary to:", summary_outfile)


################################################################################
FINAL SUMMARY
################################################################################
        feature best_order best_seasonal_order           AIC           BIC  \
0  Acousticness  (2, 1, 1)       (1, 0, 1, 12)  -1187.647236  -1162.718087   
1   Speechiness  (1, 0, 1)       (0, 0, 0, 12)  -2246.629313  -2234.083059   
2       Valence  (1, 0, 1)       (0, 0, 0, 12)  -1532.389337  -1519.843082   
3         Tempo  (1, 1, 2)       (0, 1, 1, 12)   2649.885267   2670.519612   
4      Duration  (2, 0, 2)       (1, 1, 1, 12)  10458.341764  10487.245115   

     ADF_pvalue  has_seasonal_structure  num_significant_seasonal_terms  \
0  1.889980e-03                    True                               2   
1  8.197318e-01                   False                               0   
2  5.798530e-01                   False                               0   
3  1.685555e-13                    True               